<a href="https://colab.research.google.com/github/IdrisJunaidAI/ITAI_ML_FirstRepo_IdrisJunaid/blob/main/L13_IdrisJunaid_ITAI1371.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 13 Lab: Building Machine Learning Pipelines

**Student:** Idris O. Junaid  
**Course:** ITAI 1371  
**Lab:** L13  
**Topic:** Building Machine Learning Pipelines

## Objective

The objective of this lab is to understand how `scikit learn` pipelines create cleaner, safer, reproducible, and deployment ready machine learning workflows. The lab compares a manual preprocessing workflow with a professional pipeline that combines imputation, scaling, categorical encoding, and model training in one controlled object.


## Part 1: Why Use Pipelines?

A typical machine learning workflow includes data loading, cleaning, splitting, feature preprocessing, model training, and evaluation. When these steps are managed as separate objects, the workflow becomes harder to reproduce and easier to apply incorrectly.

### Data leakage

Data leakage occurs when information from the test set influences training. For example, calculating a mean or standard deviation from the complete dataset before the train and test split gives the model indirect information about the test data. This can produce an unrealistically optimistic evaluation.

A `scikit learn` pipeline addresses this risk by:

1. Encapsulating preprocessing and modeling in one object.
2. Learning imputation, scaling, and encoding rules from the training data only.
3. Reusing the learned rules consistently when predicting on test or production data.
4. Supporting reproducible evaluation, cross validation, tuning, persistence, and deployment.


## Part 2: The Manual Method

The following code performs the workflow with separate objects. The numerical and categorical imputers, scaler, encoder, and classifier are managed individually.

To keep the comparison valid, every preprocessing object is fitted only on the training data. The test data is transformed with the statistics and categories learned from the training data.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# Load the Titanic data from the course source.
# The local path is a fallback that allows the completed notebook to run offline.
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "datasciencedojo/datasets/master/titanic.csv"
)
LOCAL_DATA_PATH = Path("/mnt/data/titanic.csv")


def load_titanic_data() -> pd.DataFrame:
    """Load the Titanic dataset from the URL or the local offline fallback."""
    try:
        return pd.read_csv(DATA_URL)
    except Exception:
        if LOCAL_DATA_PATH.exists():
            return pd.read_csv(LOCAL_DATA_PATH)
        raise


# Load the raw data and remove columns that are not used by this model.
df = load_titanic_data()
df = df.drop(columns=["Cabin", "Name", "Ticket", "PassengerId"])

X = df.drop(columns="Survived")
y = df["Survived"]

# Split before learning any preprocessing statistics.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

numeric_features = ["Age", "Fare", "SibSp", "Parch"]
categorical_features = ["Pclass", "Sex", "Embarked"]

# Learn the numerical median values from the training data only.
numeric_imputer = SimpleImputer(strategy="median")
X_train_num_imputed = numeric_imputer.fit_transform(
    X_train[numeric_features]
)
X_test_num_imputed = numeric_imputer.transform(
    X_test[numeric_features]
)

# Learn the scaling statistics from the imputed training data only.
scaler = StandardScaler()
X_train_scaled_num = scaler.fit_transform(X_train_num_imputed)
X_test_scaled_num = scaler.transform(X_test_num_imputed)

# Learn the most frequent categorical values from the training data only.
categorical_imputer = SimpleImputer(strategy="most_frequent")
X_train_cat_imputed = categorical_imputer.fit_transform(
    X_train[categorical_features]
)
X_test_cat_imputed = categorical_imputer.transform(
    X_test[categorical_features]
)

# Learn the categorical vocabulary from the training data only.
encoder = OneHotEncoder(handle_unknown="ignore")
X_train_encoded_cat = encoder.fit_transform(X_train_cat_imputed)
X_test_encoded_cat = encoder.transform(X_test_cat_imputed)

# Combine the numerical and categorical matrices for the classifier.
X_train_processed = np.hstack(
    [X_train_scaled_num, X_train_encoded_cat.toarray()]
)
X_test_processed = np.hstack(
    [X_test_scaled_num, X_test_encoded_cat.toarray()]
)

# Train and evaluate the manual workflow.
manual_model = RandomForestClassifier(random_state=42)
manual_model.fit(X_train_processed, y_train)

y_pred_manual = manual_model.predict(X_test_processed)
manual_accuracy = accuracy_score(y_test, y_pred_manual)

print(f"Training rows: {X_train.shape[0]}")
print(f"Test rows: {X_test.shape[0]}")
print(f"Processed feature count: {X_train_processed.shape[1]}")
print(f"Accuracy using the manual method: {manual_accuracy:.2%}")

Training rows: 712
Test rows: 179
Processed feature count: 12
Accuracy using the manual method: 82.68%


## Part 3: The Pipeline Method

The pipeline version performs the same logical operations while controlling the order of execution.

The numerical branch imputes missing values with the median and then standardizes the numerical variables. The categorical branch imputes the most frequent value and then applies one hot encoding. A `ColumnTransformer` sends each group of columns through the correct branch. The final pipeline connects the complete preprocessor to the `RandomForestClassifier`.

Only one call to `fit()` is required.


In [2]:
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline


# Reload the raw dataset so the pipeline receives unprocessed input.
df_pipeline = load_titanic_data()
df_pipeline = df_pipeline.drop(
    columns=["Cabin", "Name", "Ticket", "PassengerId"]
)

X_pipeline = df_pipeline.drop(columns="Survived")
y_pipeline = df_pipeline["Survived"]

X_train_pipe, X_test_pipe, y_train_pipe, y_test_pipe = train_test_split(
    X_pipeline,
    y_pipeline,
    test_size=0.20,
    random_state=42,
)

# Numerical branch: median imputation followed by standardization.
numeric_transformer = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
)

# Categorical branch: mode imputation followed by one hot encoding.
categorical_transformer = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"),
)

# Apply each preprocessing branch to its assigned columns.
preprocessor = make_column_transformer(
    (
        numeric_transformer,
        ["Age", "Fare", "SibSp", "Parch"],
    ),
    (
        categorical_transformer,
        ["Pclass", "Sex", "Embarked"],
    ),
)

# Chain preprocessing and classification into one controlled workflow.
final_pipeline = make_pipeline(
    preprocessor,
    RandomForestClassifier(random_state=42),
)

# Fit every preprocessing step and the model through one pipeline call.
final_pipeline.fit(X_train_pipe, y_train_pipe)

# Predicting automatically applies transform operations before inference.
y_pred_pipeline = final_pipeline.predict(X_test_pipe)
pipeline_accuracy = accuracy_score(y_test_pipe, y_pred_pipeline)

fitted_preprocessor = final_pipeline.named_steps["columntransformer"]
processed_test = fitted_preprocessor.transform(X_test_pipe)

print(f"Accuracy using the pipeline method: {pipeline_accuracy:.2%}")
print(
    "Manual and pipeline predictions are identical:",
    np.array_equal(y_pred_manual, y_pred_pipeline),
)
print(f"Processed feature count: {processed_test.shape[1]}")
print("Pipeline steps:", list(final_pipeline.named_steps.keys()))

Accuracy using the pipeline method: 82.68%
Manual and pipeline predictions are identical: True
Processed feature count: 12
Pipeline steps: ['columntransformer', 'randomforestclassifier']


## Reflective Knowledge Check

### 1. Code comparison

The three greatest advantages of the pipeline approach are **encapsulation**, **leakage prevention**, and **reproducibility**.

First, encapsulation places imputation, scaling, encoding, and classification inside one object. This removes the need to manage many intermediate arrays and reduces the possibility of applying the steps in the wrong order.

Second, the pipeline protects the train and test boundary. When `final_pipeline.fit(X_train, y_train)` is called, each transformer learns only from the training data. During `predict(X_test)`, the pipeline reuses those learned values without fitting again.

Third, the same complete workflow can be reused during cross validation, hyperparameter tuning, model persistence, and production inference. This makes the model easier to test, document, transfer, and reproduce.

### 2. Data leakage explained

`fit_transform()` performs two operations. It first learns parameters from the supplied data, such as the mean and standard deviation used by `StandardScaler`, and then applies the transformation. It was therefore correct to call `fit_transform()` on the training data.

The test set must use `transform()` only. This applies the training mean and training standard deviation without recalculating them from the test set. Fitting the scaler on the test set would allow test information to influence preprocessing and would make the evaluation less trustworthy. It could also place training and test observations in inconsistent feature spaces.

A pipeline handles this automatically. Calling `fit()` on the pipeline fits the preprocessing steps and classifier using the training data. Calling `predict()` on test or production data invokes only the previously learned transformations before the classifier makes predictions.

### 3. Extending the pipeline with PCA

`PCA()` should be inserted **after the `preprocessor` and before the `RandomForestClassifier`**. The conceptual order would be:

```python
final_pipeline = make_pipeline(
    preprocessor,
    PCA(n_components=desired_number),
    RandomForestClassifier(random_state=42),
)
```

This position allows PCA to receive the completed numerical feature matrix after imputation, scaling, and encoding. The classifier would then train on the principal components instead of the original processed columns. If the encoder returns a sparse matrix in a larger application, I would configure the encoder to return dense output or use `TruncatedSVD`, which is designed for sparse matrices.

### 4. Real world value

Providing another team with one fitted `final_pipeline` object is safer because the object preserves the exact preprocessing sequence, learned medians, scaling statistics, encoded category vocabulary, and trained classifier. The deployment team can send raw records to one `predict()` method and receive results produced by the same logic used during training.

Providing separate objects creates several failure points. A team member might omit an imputer, use a different column order, fit a scaler again, apply the encoder incorrectly, or send the model a feature matrix with the wrong shape. These errors can silently change predictions even when the classifier itself is unchanged.

The pipeline is therefore similar to a controlled process in pharmaceutical quality assurance. Every transformation occurs in a defined sequence, the approved parameters are preserved, and the same process is applied consistently to each new record. This improves traceability, reliability, and confidence in deployment.

## Final Reflection

The most important lesson from this lab is that a reliable machine learning system includes more than the final algorithm. Preprocessing decisions are part of the model and must be controlled with the same care as model training. The manual and pipeline approaches produced the same accuracy and identical predictions, but the pipeline required fewer operational decisions and created a much safer artifact for reuse. I now understand why pipelines are fundamental for professional machine learning, particularly when models must be validated, audited, transferred, or deployed.
